# Clusterização com K-Modes

Esta primeira versão segue o fluxo exploratório do notebook de K-means, mas usa distância de correspondência para variáveis categóricas. A severidade PHQ e as respostas que a compõem ficam fora das features para evitar vazamento do target. Idade e razão de renda/pobreza são agrupadas em faixas; o target é usado apenas para comparar a composição dos clusters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from kmodes.kmodes import KModes
from sklearn.metrics import silhouette_score

ID_COLUNA = "ID_DOMICILIO"
PESO_COLUNA = "PESO_AMOSTRAL"
COLUNA_SEVERIDADE = "SEVERIDADE_DEPRESSAO_PHQ"
TARGET = "DEPRESSAO_PHQ_SIMPLIFICADO"

COLUNAS_SINTOMAS_DEPRESSAO = [
    "FREQ_PERDA_INTERESSE_2SEM",
    "FREQ_SENTIU_DEPRIMIDO_2SEM",
    "FREQ_DIFICULDADE_DORMIR_2SEM",
    "FREQ_CANSASO_2SEM",
    "FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM",
    "FREQ_AUTOAVALIACAO_NEGTIVA_2SEM",
    "FREQ_DIFICULDADE_CONCENTRACAO_2SEM",
    "MODIFICACAO_VELOCIDADE_FALA",
    "FREQ_DEPRESSAO",
    "REMEDIO_DEPRESSAO",
    "HISTORICO_DEPRESSAO",
    "INTENSIDADE_DEPRESSAO",
]

# 1. Carregar a base e excluir severidades indefinidas antes de derivar o target.
df = pd.read_csv(Path("../4_random_forest") / "adult25_limpo.csv")
df = df.set_index(ID_COLUNA)
df = df[~df[COLUNA_SEVERIDADE].isin([-1, 5])].copy()
df[TARGET] = (df[COLUNA_SEVERIDADE] != 1).astype(int)

# 2. Separar target e peso amostral das features; o peso não é usado pelo K-Modes.
y = df[TARGET]
sample_weights = df[PESO_COLUNA]
colunas_remover = [TARGET, PESO_COLUNA, COLUNA_SEVERIDADE] + COLUNAS_SINTOMAS_DEPRESSAO
X = df.drop(columns=[coluna for coluna in colunas_remover if coluna in df.columns])

print(f"Registros para clusterização: {len(X):,}")
print(f"Features antes da preparação: {X.shape[1]}")
print("Distribuição do target (usado somente para análise):")
print(y.value_counts(normalize=True).sort_index().mul(100).round(1).rename("percentual"))

## Preparar variáveis categóricas

O K-Modes não usa padronização nem one-hot encoding: cada coluna representa uma categoria. As duas variáveis contínuas usadas como numéricas no K-means são discretizadas para evitar uma categoria diferente para quase cada valor.

In [ ]:
X_categorico = X.copy()

faixas_idade = ["0-17", "18-29", "30-44", "45-59", "60-74", "75+"]
idade_valida = X_categorico["IDADE"].where(X_categorico["IDADE"] >= 0)
X_categorico["IDADE_FAIXA"] = pd.cut(
    idade_valida,
    bins=[-1, 17, 29, 44, 59, 74, np.inf],
    labels=faixas_idade,
).astype("string").fillna("Não informado")

faixas_renda = ["<1", "1-<2", "2-<3", "3-<4", "4-<5", "5+"]
X_categorico["RAZAO_RENDA_POBREZA_FAIXA"] = pd.cut(
    X_categorico["RAZAO_RENDA_POBREZA"],
    bins=[-np.inf, 1, 2, 3, 4, 5, np.inf],
    labels=faixas_renda,
    right=False,
).astype("string").fillna("Não informado")

X_categorico = X_categorico.drop(columns=["IDADE", "RAZAO_RENDA_POBREZA"])
X_categorico = X_categorico.astype("string").fillna("Não informado")

print(f"Features categóricas: {X_categorico.shape[1]}")
print(f"Valores ausentes após preparação: {int(X_categorico.isna().sum().sum())}")
X_categorico.head()

## Diagnóstico inicial do número de clusters

A curva de custo compara o total de dissimilaridades (discordâncias) para diferentes valores de *k*. O silhouette é calculado com distância de Hamming em uma amostra para controlar o custo computacional. Para tornar a exploração mais rápida, os diagnósticos usam até 5.000 registros; a avaliação final abaixo ajusta o modelo escolhido em todos os registros.

In [ ]:
def ajustar_kmodes(n_clusters, dados):
    modelo = KModes(
        n_clusters=n_clusters,
        init="Cao",
        n_init=3,
        max_iter=100,
        random_state=42,
        verbose=0,
    )
    labels = pd.Series(
        modelo.fit_predict(dados.to_numpy(dtype=str)),
        index=dados.index,
        name="cluster",
    )
    return modelo, labels, modelo.cost_


X_amostra = X_categorico.sample(n=min(5000, len(X_categorico)), random_state=42)
X_amostra_codificada = X_amostra.apply(lambda coluna: pd.factorize(coluna, sort=True)[0])

ks = range(2, 7)
custos = []
silhouettes = []

for k in ks:
    modelo_amostra, labels_amostra, custo = ajustar_kmodes(k, X_amostra)
    custos.append(custo)
    indices_silhouette = X_amostra.sample(
        n=min(1000, len(X_amostra)), random_state=42
    ).index
    dados_silhouette = X_amostra_codificada.loc[indices_silhouette]
    labels_silhouette = labels_amostra.loc[indices_silhouette]
    if 1 < labels_silhouette.nunique() < len(labels_silhouette):
        silhouette = silhouette_score(
            dados_silhouette,
            labels_silhouette,
            metric="hamming",
        )
    else:
        silhouette = np.nan
    silhouettes.append(silhouette)

pd.DataFrame({"k": list(ks), "custo": custos, "silhouette_hamming": silhouettes})

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(list(ks), custos, "bo-")
ax1.set(title="Custo do K-Modes (amostra)", xlabel="Número de clusters (k)", ylabel="Dissimilaridade total")
ax2.plot(list(ks), silhouettes, "go-")
ax2.set(title="Silhouette com distância de Hamming", xlabel="Número de clusters (k)", ylabel="Silhouette")
plt.tight_layout()
plt.show()

## Ajuste e leitura dos clusters

Comece com `K_FINAL = 3` como hipótese inicial e ajuste após observar custo e silhouette. O target não participa da formação dos clusters; sua composição é exibida apenas para interpretação posterior.

In [ ]:
K_FINAL = 3
km, labels, custo_final = ajustar_kmodes(K_FINAL, X_categorico)

resultado = pd.DataFrame({"target": y, "cluster": labels})
tamanho_clusters = labels.value_counts().sort_index().rename("registros")
print(f"Custo final: {custo_final:,.0f}")
print("Tamanho dos clusters:")
print(tamanho_clusters)

contagens_target = pd.crosstab(resultado["cluster"], resultado["target"])
proporcoes_target = contagens_target.div(contagens_target.sum(axis=1), axis=0).mul(100)
print("\nComposição do target (%) dentro de cada cluster:")
print(proporcoes_target.round(1))

ax = proporcoes_target.plot(kind="bar", stacked=True, figsize=(8, 5), colormap="Set2")
ax.set(title="Composição do target em cada cluster", xlabel="Cluster", ylabel="Proporção (%)")
ax.set_ylim(0, 100)
ax.legend(title=TARGET)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
colunas_perfil = [
    "IDADE_FAIXA",
    "RAZAO_RENDA_POBREZA_FAIXA",
    "SEXO",
    "REGIAO_GEOGRAFICA",
    "SUPORTE_SOCIAL",
    "AUTOAVALIACAO_SAUDE_GERAL",
    "CATEGORIA_IMC",
    "HISTORICO_TRANSTORNO_ANSIEDADE",
]
colunas_perfil = [coluna for coluna in colunas_perfil if coluna in X_categorico.columns]
X_cluster = X_categorico.join(labels)

def primeira_moda(coluna):
    modas = coluna.mode()
    return modas.iloc[0] if not modas.empty else "Não informado"

perfil_clusters = X_cluster.groupby("cluster")[colunas_perfil].agg(primeira_moda)
perfil_clusters.insert(0, "registros", tamanho_clusters)
print("Categorias mais frequentes em cada cluster:")
perfil_clusters